# 05 - Build A Dual-Provider RAG Pipeline

## Learning objectives

- Build a minimal RAG pipeline end to end.
- Retrieve evidence once and send the same context to OpenAI and Claude.
- Produce grounded answers with citations.
- Compare answer style, grounding, and provider behavior.

## Concept primer

RAG is not just `put documents into the prompt`. A reliable RAG pipeline has explicit stages: chunk, embed, retrieve, assemble prompt, generate answer, and evaluate whether the answer is grounded.


## Step 1 - Setup


In [ ]:
# This setup cell keeps imports working whether Jupyter starts in the repo root
# or inside the nlp_transformers_embeddings folder.
from pathlib import Path
import os
import sys

CURRENT = Path.cwd()
COURSE_ROOT = CURRENT.parent if CURRENT.name == "nlp_transformers_embeddings" else CURRENT
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dotenv import load_dotenv
load_dotenv(COURSE_ROOT / ".env", override=False)
load_dotenv(COURSE_ROOT / "nlp_transformers_embeddings" / ".env", override=False)

# Security practice: print whether keys are present, never the key values.
print("LIVE_API enabled:", os.getenv("LIVE_API", "false").lower() == "true")
print("OPENAI_API_KEY loaded:", bool(os.getenv("OPENAI_API_KEY")))
print("ANTHROPIC_API_KEY loaded:", bool(os.getenv("ANTHROPIC_API_KEY")))


## Step 2 - Build the retrieval stage


In [ ]:
from nlp_transformers_embeddings.utils.mock_data import SAMPLE_DOCUMENTS
from nlp_transformers_embeddings.utils.openai_client import embed_texts, generate_text
from nlp_transformers_embeddings.utils.anthropic_client import claude_message
from nlp_transformers_embeddings.utils.retrieval import chunk_documents, rank_chunks
from nlp_transformers_embeddings.utils.schemas import GroundedAnswer

question = "How should students securely use API keys while learning RAG?"

# 1. Chunk source documents so each passage can fit into a prompt and be cited.
chunks = chunk_documents(SAMPLE_DOCUMENTS, chunk_size_words=50, overlap_words=10)

# 2. Embed chunks and the question. OpenAI is the first-party embedding path.
chunk_embeddings = embed_texts([chunk.text for chunk in chunks])
question_embedding = embed_texts([question])[0]

# 3. Retrieve the top passages before generation.
retrieved = rank_chunks(question_embedding, chunk_embeddings, chunks, top_k=3)

for item in retrieved:
    print(f"[{item.rank}] {item.chunk.title} | score={item.score:.3f}")
    print(item.chunk.text + "\n")


## Step 3 - Assemble a grounded prompt

The prompt tells the model to answer only from evidence and cite passage IDs. This is one of the most important RAG habits students should practice.


In [ ]:
def build_rag_prompt(question, passages):
    # Passage numbers become citation handles the model can reference in its answer.
    evidence = "\n\n".join(
        f"[{item.rank}] Title: {item.chunk.title}\n{item.chunk.text}"
        for item in passages
    )
    return f"""Answer the question using only the evidence below.
If the evidence is insufficient, say what is missing.
Cite passages with bracketed numbers like [1].

Question: {question}

Evidence:
{evidence}

Grounded answer:"""

rag_prompt = build_rag_prompt(question, retrieved)
print(rag_prompt[:900] + "...")


## Step 4 - Generate with OpenAI and Claude

Both providers receive the same retrieved evidence. This gives students an apples-to-apples comparison of answer behavior, even though OpenAI was used for embeddings.


In [ ]:
openai_answer_text = generate_text(
    rag_prompt,
    instructions="You are a careful RAG assistant. Cite evidence and avoid unsupported claims.",
    mock_response="Students should store API keys in a local .env file, avoid printing key values, and never commit credentials to GitHub or shared notebooks [1]. They should run mock mode when credentials are missing or when avoiding paid calls [1].",
)

claude_answer_text = claude_message(
    rag_prompt,
    system="You are a careful RAG assistant. Cite evidence and avoid unsupported claims.",
    mock_response="Students should treat API keys like passwords: keep them in a local .env file, never paste them into shared notebooks, and never commit them to GitHub [1]. A safe notebook should show only whether a key is loaded, not the secret value [1].",
)

answers = [
    GroundedAnswer(provider="OpenAI", question=question, answer=openai_answer_text, citations=["1"]),
    GroundedAnswer(provider="Anthropic", question=question, answer=claude_answer_text, citations=["1"]),
]

for answer in answers:
    print(f"\n{answer.provider} answer:\n{answer.answer}")


## What just happened?

The RAG system used one retrieval layer and two generation layers. This is the architecture pattern that will make future Google AI or Hugging Face additions straightforward: add a new generation adapter, keep the retrieval contract stable.


## Student exercise

Change the question to one that the corpus cannot answer. A well-grounded assistant should say what evidence is missing instead of guessing.

## Debugging checklist

- If citations are missing, tighten the prompt and require bracketed passage numbers.
- If the answer includes unsupported claims, add an evaluator step before showing it to users.
- If retrieval returns the wrong passage, debug embeddings and chunking before blaming the generator.

## Production best practices

- Store retrieved chunk IDs with the answer for auditability.
- Add refusal behavior for insufficient evidence.
- Track latency and cost separately for embedding, retrieval, and generation.
- Evaluate provider outputs on the same evidence set.

## Reflection questions

1. Why is shared retrieval useful for comparing OpenAI and Claude?
2. What makes an answer grounded rather than merely plausible?
3. What should the system do when no retrieved passage is relevant?
